# 04. Context compaction

Сравниваем ответы на **полном** retrieved-контексте и на **сжатом**.

Сжатие extractive: из каждого чанка выбираются самые релевантные вопросу
предложения. Формулировки сохраняются, новых фактов не появляется.

Дизайн:

- retrieval фиксирован: top-5;
- full context — все 5 чанков;
- compact context — те же 5 чанков, но сжатые до бюджета 500 токенов;
- ответы генерирует DeepSeek;
- judge оценивает correctness / faithfulness / citations по шкале 1–5.

Главные метрики: контекстные токены и качество ответа.


In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo:", ROOT)


repo: D:\diffeq-wiki-rag


In [2]:
import json
import time

import pandas as pd

from app.llm import get_llm
from app.rag.compact import SentenceScorer, compact_documents
from app.rag.pipeline import format_context
from app.rag.store import build_vectorstore
from eval.metrics import load_golden
from eval.rag_eval import (
    answer_with_context,
    build_answer_chain,
    citation_stats,
    judge_answer,
)

items = load_golden()
store = build_vectorstore()
scorer = SentenceScorer()
llm = get_llm()
chain = build_answer_chain(llm)
print("вопросов:", len(items))


D:\diffeq-wiki-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   9%|▊         | 17/199 [00:00<00:01, 164.40it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1322.60it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5189.55it/s]

Default prompt name is set to 'passage'. This prompt applies unless overridden by `prompt`, `prompt_name`, or a method-specific prompt, including an empty prompt.


вопросов: 30


## 1. Генерация и оценка

Для каждого вопроса:

1. retrieve top-5;
2. full context;
3. compact context (500 токенов);
4. два ответа DeepSeek;
5. два вызова judge.

Это 30 × (2 ответа + 2 judge) = 120 вызовов, поэтому ноутбук идёт несколько минут.


In [3]:
rows = []
started = time.perf_counter()


def generate(question: str, context: str) -> str:
    last_error = None
    for _ in range(3):
        try:
            return answer_with_context(chain, question, context)
        except Exception as exc:
            last_error = exc
            time.sleep(1.5)
    return f"ERROR: {type(last_error).__name__}"


for index, item in enumerate(items, start=1):
    question = item["question"]
    hits = store.similarity_search(question, k=5)

    compacted, stats = compact_documents(
        hits,
        question,
        scorer,
        max_tokens=500,
    )
    full_context = format_context(hits)
    compact_context = format_context(compacted)

    answer_full = generate(question, full_context)
    answer_compact = generate(question, compact_context)

    judge_full = judge_answer(
        question,
        item["reference_answer"],
        item["key_facts"],
        full_context,
        answer_full,
        llm=llm,
    )
    judge_compact = judge_answer(
        question,
        item["reference_answer"],
        item["key_facts"],
        compact_context,
        answer_compact,
        llm=llm,
    )

    rows.append(
        {
            "id": item["id"],
            "topic": item["topic"],
            "question": question,
            "tokens_full": stats.before_tokens,
            "tokens_compact": stats.after_tokens,
            "reduction_pct": round(
                100 * (1 - stats.after_tokens / stats.before_tokens), 1
            ),
            "answer_full": answer_full,
            "answer_compact": answer_compact,
            "judge_full": judge_full,
            "judge_compact": judge_compact,
            "citations_full": citation_stats(answer_full, len(hits)),
            "citations_compact": citation_stats(answer_compact, len(hits)),
        }
    )

    elapsed = time.perf_counter() - started
    print(
        f"[{index}/{len(items)}] {item['id']} | "
        f"tokens {stats.before_tokens}->{stats.after_tokens} | "
        f"elapsed {elapsed:.0f}s"
    )
    time.sleep(0.15)


[1/30] lipschitz-def-01 | tokens 1253->500 | elapsed 10s


[2/30] lipschitz-example-02 | tokens 981->500 | elapsed 19s


[3/30] lipschitz-props-03 | tokens 1047->496 | elapsed 30s


[4/30] lipschitz-picard-04 | tokens 930->499 | elapsed 39s


[5/30] linearode-general-05 | tokens 837->492 | elapsed 47s


[6/30] linearode-homconst-06 | tokens 899->494 | elapsed 55s


... (20 outputs left out)

[27/30] multi-wronskian-ode-27 | tokens 1099->500 | elapsed 247s


[28/30] multi-integrating-linear-28 | tokens 1193->499 | elapsed 255s


[29/30] multi-charpoly-basis-29 | tokens 1158->496 | elapsed 264s


[30/30] multi-reduction-system-30 | tokens 1335->500 | elapsed 273s


## 2. Сводные метрики

In [4]:
def mean_judge(rows, key, field):
    values = [
        row[key][field]
        for row in rows
        if row.get(key) and row[key].get(field) is not None
    ]
    return round(sum(values) / len(values), 3) if values else None


summary_rows = []
for mode in ["full", "compact"]:
    judge_key = f"judge_{mode}"
    citation_key = f"citations_{mode}"
    summary_rows.append(
        {
            "mode": mode,
            "context_tokens_mean": round(
                sum(row[f"tokens_{mode}"] for row in rows) / len(rows), 1
            ),
            "correctness": mean_judge(rows, judge_key, "correctness"),
            "faithfulness": mean_judge(rows, judge_key, "faithfulness"),
            "citations_score": mean_judge(rows, judge_key, "citations"),
            "has_citations_pct": round(
                100 * sum(row[citation_key]["has_citations"] for row in rows) / len(rows), 1
            ),
            "valid_citation_ratio": round(
                sum(row[citation_key]["valid_ratio"] for row in rows) / len(rows), 3
            ),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df


,mode,context_tokens_mean,correctness,faithfulness,citations_score,has_citations_pct,valid_citation_ratio
0,full,1072.3,3.733,4.867,4.733,100.0,1.0
1,compact,497.7,3.500,4.733,4.633,100.0,1.0


## 3. Сжатие контекста

In [5]:
token_df = pd.DataFrame(
    [
        {
            "tokens_full_mean": round(sum(row["tokens_full"] for row in rows) / len(rows), 1),
            "tokens_compact_mean": round(sum(row["tokens_compact"] for row in rows) / len(rows), 1),
            "reduction_pct_mean": round(sum(row["reduction_pct"] for row in rows) / len(rows), 1),
            "tokens_full_max": max(row["tokens_full"] for row in rows),
            "tokens_compact_max": max(row["tokens_compact"] for row in rows),
        }
    ]
)
token_df


,tokens_full_mean,tokens_compact_mean,reduction_pct_mean,tokens_full_max,tokens_compact_max
0,1072.3,497.7,52.1,1466,500


## 4. По вопросам

In [6]:
per_question = pd.DataFrame(
    [
        {
            "id": row["id"],
            "tokens_full": row["tokens_full"],
            "tokens_compact": row["tokens_compact"],
            "reduction_pct": row["reduction_pct"],
            "correctness_full": (row["judge_full"] or {}).get("correctness"),
            "correctness_compact": (row["judge_compact"] or {}).get("correctness"),
            "faithfulness_full": (row["judge_full"] or {}).get("faithfulness"),
            "faithfulness_compact": (row["judge_compact"] or {}).get("faithfulness"),
            "citations_full": (row["judge_full"] or {}).get("citations"),
            "citations_compact": (row["judge_compact"] or {}).get("citations"),
        }
        for row in rows
    ]
)
per_question


,id,tokens_full,tokens_compact,reduction_pct,correctness_full,correctness_compact,faithfulness_full,faithfulness_compact,citations_full,citations_compact
0,lipschitz-def-01,1253,500,60.1,4,5,5,5,5,5
1,lipschitz-example-02,981,500,49.0,2,2,4,4,4,4
2,lipschitz-props-03,1047,496,52.6,2,2,5,5,5,5
3,lipschitz-picard-04,930,499,46.3,4,4,4,5,4,5
4,linearode-general-05,837,492,41.2,3,3,5,5,5,5
5,linearode-homconst-06,899,494,45.1,2,2,4,4,4,4
6,linearode-charpoly-07,1380,500,63.8,5,2,5,4,5,4
7,linearode-nonhom-08,874,498,43.0,5,4,5,5,5,5
8,linearode-system-09,1121,500,55.4,4,3,5,4,5,4
9,ode-def-10,1255,500,60.2,1,2,5,5,4,4


## 5. Сохранение

In [7]:
payload = {
    "summary": summary_df.to_dict(orient="records"),
    "tokens": token_df.to_dict(orient="records"),
    "rows": rows,
}
output = Path("eval/results/context_compaction_experiment.json")
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(
    json.dumps(payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("сохранено:", output)


сохранено: eval\results\context_compaction_experiment.json


## 6. Как читать

- `context_tokens_mean` — сколько токенов уходит в LLM;
- `correctness` — соответствие эталонному ответу и ключевым фактам;
- `faithfulness` — подтверждаются ли утверждения контекстом;
- `citations_score` — корректность ссылок `[n]`;
- `has_citations_pct` — доля ответов со ссылками.

Если compact снижает токены, но не ухудшает correctness/faithfulness — сжатие
удачное. Если faithfulness падает, значит extractive-сжатие выбросило важные
предложения.


## 7. Выводы

### Качество ответа

| Режим | correctness | faithfulness | citations | ответов со ссылками |
|---|---|---|---|---|
| full | 3.73 | 4.87 | 4.73 | 100% |
| compact | 3.50 | 4.73 | 4.63 | 100% |

### Стоимость

| Режим | context tokens | reduction |
|---|---|---|
| full | 1072.3 | — |
| compact | **497.7** | **52.1%** |

### Correctness по темам

| Тема | full | compact | Δ |
|---|---|---|---|
| integrating factor | 4.67 | 4.67 | 0.00 |
| linear ODE | 3.80 | 2.80 | −1.00 |
| lipschitz | 3.00 | 3.25 | +0.25 |
| multi-hop | 4.20 | 3.80 | −0.40 |
| ODE | 3.50 | 3.25 | −0.25 |
| Picard | 2.60 | 2.60 | 0.00 |
| Wronskian | 4.75 | 4.75 | 0.00 |

### Token reduction по темам

| Тема | reduction |
|---|---|
| integrating factor | 62% |
| linear ODE | 50% |
| lipschitz | 52% |
| multi-hop | 54% |
| ODE | 50% |
| Picard | 47% |
| Wronskian | 54% |

Сжатие уменьшило контекст на **52.1%** (1072 → 498 токенов).
Качество просело умеренно: correctness −0.23, faithfulness −0.13,
citations −0.10. Все ответы по-прежнему содержат корректные ссылки.

Главная просадка — `linearode-charpoly-07` (5 → 2): это формулоёмкий раздел,
и extractive-сжатие выбросило важные предложения. На Lipschitz и ODE сжатие,
наоборот, немного улучшило ответы — меньше шума.

Вывод: extractive compaction жизнеспособен, но бюджет 500 токенов слишком
агрессивен для формулоёмких разделов. Следующие шаги:

1. sweep бюджета: 500 / 700 / 900 токенов;
2. formula-aware splitting: не разрывать формулы и определения;
3. сохранять предложения с формулами и теоремами принудительно;
4. затем — рефакторинг агента и answer-level сравнение RAG vs agent.
